# Retail-Hype Intraday Execution Under Capital and Liquidity Constraints

Trading final project notebook. Setup-first, ML-second, IBKR paper execution last.

The project starts with named hypotheses for SOUN, GME, AMC, RKLB, LUNR, ACHR, BBAI, KOSS, and RIVN. The model only gates those setups; it does not search the whole market.


## Research question

Can a small-capital trader improve intraday paper-trading results in retail-hype names by combining named setups, time windows, liquidity/cost gates, and a simple ML gate?


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
UNIVERSE = ROOT / 'data/final_project_universe.csv'
SETUPS = ROOT / 'data/final_project_trade_setups.csv'

universe = pd.read_csv(UNIVERSE)
setups = pd.read_csv(SETUPS)
universe


In [ ]:
setups[['setup_id','symbol','family','setup_name','research_side','live_side_allowed','entry_window_et','thesis']]


## Data download

Run this in terminal from the repo root:

```bash
python research/download_prices.py --universe data/final_project_universe.csv --period 60d --interval 5m
python research/build_features.py --policy config/model_policy.example.toml
python research/apply_trade_setups.py --features data/features/features.csv --setups data/final_project_trade_setups.csv --out data/reports/setup_occurrences.csv
```


In [ ]:
features_path = ROOT / 'data/features/features.csv'
if features_path.exists():
    features = pd.read_csv(features_path)
    display(features.head())
    print(features.shape)
else:
    print('No feature file yet. Run the download/build commands above.')


In [ ]:
occ_path = ROOT / 'data/reports/setup_occurrences.csv'
if occ_path.exists():
    occ = pd.read_csv(occ_path)
    display(occ.head(20))
    print(occ.groupby(['symbol','setup_id']).size().sort_values(ascending=False))
else:
    print('No setup occurrence file yet. Run research/apply_trade_setups.py first.')


## Setup-only baseline

The baseline is not ML. It asks: when the hand-written setup fires, what is the forward return after a simple cost buffer?


In [ ]:
if occ_path.exists():
    occ = pd.read_csv(occ_path)
    if 'forward_return' in occ.columns:
        cost_buffer = 0.0035  # 35 bps placeholder; replace with measured spread+slippage
        occ['net_forward_return'] = occ['forward_return'] - cost_buffer
        summary = occ.groupby(['setup_id','symbol']).agg(
            n=('net_forward_return','size'),
            avg_net_return=('net_forward_return','mean'),
            hit_rate=('net_forward_return', lambda x: (x > 0).mean()),
            avg_score=('setup_score','mean'),
        ).reset_index().sort_values('avg_net_return', ascending=False)
        display(summary)
    else:
        print('forward_return not found in setup file.')


## ML gate

The ML task is binary: given that a named setup appeared, predict whether the next 3 bars beat estimated costs. Use date-based train/test split, not a random row split.


In [ ]:
if occ_path.exists():
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    occ = pd.read_csv(occ_path)
    feature_cols = ['ret_1','ret_3','ret_12','range_bps','rel_volume_20','realized_vol_20','close_vs_sma20_bps']
    needed = feature_cols + ['label_forward_up','datetime']
    if set(needed).issubset(occ.columns) and len(occ) > 30:
        occ['datetime'] = pd.to_datetime(occ['datetime'], utc=True)
        occ = occ.sort_values('datetime').dropna(subset=needed)
        split = int(len(occ) * 0.70)
        train, test = occ.iloc[:split], occ.iloc[split:]
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        model.fit(train[feature_cols], train['label_forward_up'])
        prob = model.predict_proba(test[feature_cols])[:,1]
        pred = (prob >= 0.55).astype(int)
        print(classification_report(test['label_forward_up'], pred, zero_division=0))
        if test['label_forward_up'].nunique() > 1:
            print('AUC:', roc_auc_score(test['label_forward_up'], prob))
    else:
        print('Need more setup rows before training the ML gate.')


## IBKR paper execution

Only after a setup passes the project rules do we export inactive intent rows. Review manually, then paper trade through the guarded IBKR layer.

```bash
python research/export_trade_intents_from_setups.py --setup-occurrences data/reports/setup_occurrences.csv --out data/intents.generated.csv --mode paper
python -m ibkr_microexec.cli review-intents --config config/meme_stock_trading_plan.example.toml --intents data/intents.generated.csv
```


## Final tables to include

1. Universe table
2. Setup definitions
3. Setup-only baseline performance
4. ML-gated performance
5. Spread/liquidity rejection table
6. Paper-trade fill table
7. Time-window table
8. Failure-case examples
